# FinFlowML Stage 4 adapter benchmark

This notebook evaluates the already-trained adapter only. It never imports or calls the training runner. It compares the pinned Qwen base model and the read-only LoRA adapter on the official 347-document SROIE test split using identical prompts, decoding, parsing, normalization, and metrics.

Verified dataset commit: `f845db1c2ccaf883550320fe450d1e723374be32`. The repository did not preserve the original training dataset revision, so this is recorded as the verified `mp-02/sroie` dataset HEAD, not asserted to be the historical training snapshot.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ADAPTER_PATH = '/content/drive/MyDrive/FinFlowML-stage4-final-v2/best_adapter'
DATASET_REVISION = 'f845db1c2ccaf883550320fe450d1e723374be32'
OUTPUT_ROOT = '/content/drive/MyDrive/FinFlowML-stage4-evaluations'
RESUME_OUTPUT = None  # Set to an existing run directory after a disconnect
print(ADAPTER_PATH, DATASET_REVISION, OUTPUT_ROOT)

## Checkout the verified source

The checkout is pinned to the exact commit containing the evaluation runner. This notebook does not use the working tree or a source ZIP.

In [ ]:
import os

WORKTREE = '/content/FinFlowML-stage4-eval'
!rm -rf /content/FinFlowML-stage4-eval
!git clone https://github.com/Tanz2024/FinFlowML.git /content/FinFlowML-stage4-eval
os.chdir(WORKTREE)
!git checkout --detach d0c753e7b04fa3ea1bd9333d0592c6b9da1a8680
assert os.path.isfile('ml/evaluation/stage4_adapter_benchmark.py')
print('Prepared source:', os.getcwd())

In [ ]:
!pip install -q -r requirements-training.txt

import importlib.metadata as metadata

for name in ('torch', 'transformers', 'peft', 'bitsandbytes', 'datasets', 'safetensors'):
    print(name, metadata.version(name))

In [ ]:
import torch

!nvidia-smi --query-gpu=name,memory.total,memory.free,memory.used --format=csv,noheader
assert torch.cuda.is_available(), 'CUDA GPU is required; stop without running evaluation'
props = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print({'name': props.name, 'total_gib': round(total / 2**30, 2), 'free_gib': round(free / 2**30, 2), 'bf16': torch.cuda.is_bf16_supported()})
assert free >= 8 * 2**30, 'Less than 8 GiB currently free; restart/clean the runtime before continuing'

In [ ]:
import json
import os

assert os.path.isdir(ADAPTER_PATH), ADAPTER_PATH
assert os.path.isfile(os.path.join(ADAPTER_PATH, 'adapter_config.json'))
assert any(os.path.isfile(os.path.join(ADAPTER_PATH, name)) for name in ('adapter_model.safetensors', 'adapter_model.bin'))
assert not os.path.realpath(OUTPUT_ROOT).startswith(os.path.realpath(ADAPTER_PATH))
print('Read-only adapter preflight passed:', ADAPTER_PATH)
print(json.load(open(os.path.join(ADAPTER_PATH, 'adapter_config.json'))))

In [ ]:
from datetime import UTC, datetime

run_id = datetime.now(UTC).strftime('run-%Y%m%dT%H%M%SZ')
EVAL_OUTPUT = RESUME_OUTPUT or os.path.join(OUTPUT_ROOT, run_id)
RESUME_FLAG = '--resume' if RESUME_OUTPUT else ''
if not RESUME_OUTPUT and os.path.exists(EVAL_OUTPUT):
    raise RuntimeError(f'Refusing to overwrite existing output: {EVAL_OUTPUT}')
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('Evaluation output:', EVAL_OUTPUT, 'resume:', bool(RESUME_OUTPUT))

In [ ]:
!python -m ml.evaluation.stage4_adapter_benchmark \
  --adapter "{ADAPTER_PATH}" \
  --output "{EVAL_OUTPUT}" \
  --dataset-revision "{DATASET_REVISION}" \
  --max-new-tokens 192 {RESUME_FLAG}

In [ ]:
report_path = os.path.join(EVAL_OUTPUT, 'report.json')
report = json.load(open(report_path))
assert report['dataset_integrity']['counts'] == {'train': 563, 'validation': 63, 'test': 347}
assert report['dataset_integrity']['dataset_revision'] == DATASET_REVISION
assert not any(report['dataset_integrity']['overlap']['exact_fingerprint_overlap'].values())
assert report['base_summary']['documents'] == 347
assert report['adapter_summary']['documents'] == 347
assert len(report['paired_documents']) == 347
print(json.dumps({'output': EVAL_OUTPUT, 'base': report['base_summary'], 'adapter': report['adapter_summary'], 'paired': report['paired_summary']}, indent=2))